# Phase 1 — Lesson 17: Linear Systems
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 17: Linear Systems
**Type:** Build | **Languages:** Python | **Time:** ~120 minutes
**Prerequisites:** Phase 1, Lessons 01, 02, 03



# What You'll Learn

---

## Learning Objectives

- Solve Ax = b using Gaussian elimination with partial pivoting and back substitution
- Factor matrices with LU, QR, and Cholesky decompositions
- Derive the normal equations for least squares and connect to linear/ridge regression
- Diagnose ill-conditioned systems using the condition number


# The Problem

---

Every linear regression solves Ax = b. A is your data matrix, b is your target, x is the weights.

Three outcomes:
- **Unique solution**: A is invertible
- **No solution**: overdetermined (more equations than unknowns) → least squares
- **Infinite solutions**: A has a null space


In [ ]:
import numpy as npdef gaussian_eliminate(A, b):    A = A.astype(float).copy(); b = b.astype(float).copy()    n = len(b)    for col in range(n):        max_row = col + np.argmax(np.abs(A[col:, col]))        A[[col, max_row]] = A[[max_row, col]]        b[[col, max_row]] = b[[max_row, col]]        pivot = A[col, col]        for row in range(col + 1, n):            factor = A[row, col] / pivot            A[row, col:] -= factor * A[col, col:]            b[row] -= factor * b[col]    x = np.zeros(n)    for i in range(n - 1, -1, -1):        x[i] = (b[i] - A[i, i+1:] @ x[i+1:]) / A[i, i]    return xA = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)b = np.array([8, -11, -3], dtype=float)x = gaussian_eliminate(A, b)print(f"Solution: x = {x}")print(f"Verify Ax = b: {A @ x}")

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Condition number demoA_well = np.array([[2, 0], [0, 1]], dtype=float)A_ill = np.array([[1, 1], [1, 1 + 1e-10]], dtype=float)print(f"Well-conditioned: cond = {np.linalg.cond(A_well):.1f}")print(f"Ill-conditioned:  cond = {np.linalg.cond(A_ill):.2e}")b = np.array([2, 2], dtype=float)x_well = np.linalg.solve(A_well, b)x_ill = np.linalg.solve(A_ill, b)print(f"\nWell-conditioned solution: {x_well}")print(f"Ill-conditioned solution:  {x_ill}")# Perturb b slightlyb_pert = b + np.array([1e-8, -1e-8])x_ill_pert = np.linalg.solve(A_ill, b_pert)print(f"\nPerturbed ill-conditioned solution: {x_ill_pert}")print(f"Change: {np.linalg.norm(x_ill - x_ill_pert):.2e}")

# LU Decomposition

---

Factor A = L·U. Once factored, solving Ax = b for any new b costs O(n²) instead of O(n³).

With partial pivoting: PA = LU.


In [ ]:
import numpy as npdef lu_decompose(A):    n = A.shape[0]    L = np.eye(n)    U = A.astype(float).copy()    P = np.eye(n)    for k in range(n):        max_row = k + np.argmax(np.abs(U[k:, k]))        U[[k, max_row]] = U[[max_row, k]]        P[[k, max_row]] = P[[max_row, k]]        if k > 0:            L[[k, max_row], :k] = L[[max_row, k], :k]        for i in range(k + 1, n):            L[i, k] = U[i, k] / U[k, k]            U[i, k:] -= L[i, k] * U[k, k:]    return P, L, UA = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]], dtype=float)b = np.array([8, -11, -3], dtype=float)P, L, U = lu_decompose(A)# Solve via LU: Ly = Pb, then Ux = yy = np.linalg.solve(L, P @ b)x = np.linalg.solve(U, y)print(f"LU solution: x = {x}")print(f"L:\n{L}")print(f"U:\n{U}")

# Cholesky Decomposition

---

For symmetric positive definite A: A = L·Lᵀ. Twice as fast as LU.

Used for: covariance matrices, Gaussian processes, ridge regression.


In [ ]:
import numpy as npdef cholesky(A):    n = A.shape[0]    L = np.zeros_like(A, dtype=float)    for i in range(n):        for j in range(i + 1):            s = A[i, j] - L[i, :j] @ L[j, :j]            if i == j:                L[i, j] = np.sqrt(s)            else:                L[i, j] = s / L[j, j]    return LA = np.array([[4, 2], [2, 5]], dtype=float)L = cholesky(A)print(f"L:\n{L}")print(f"L @ L.T:\n{L @ L.T}")print(f"Original A:\n{A}")

# Least Squares: Normal Equations

---

For overdetermined Ax = b, minimize ||Ax − b||²:

```
AᵀA x = Aᵀ b
```

This IS the closed-form solution to linear regression.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)X_raw = np.random.randn(100, 2)w_true = np.array([2.0, -1.0])y = X_raw @ w_true + np.random.randn(100) * 0.1X = np.column_stack([np.ones(100), X_raw])# Normal equations: w = (XᵀX)⁻¹ Xᵀ yw_ols = np.linalg.solve(X.T @ X, X.T @ y)w_np = np.linalg.lstsq(X, y, rcond=None)[0]print(f"OLS (normal eq):  {w_ols}")print(f"OLS (lstsq):      {w_np}")print(f"True:             {np.concatenate([[0], w_true])}")# Plot regression linex_plot = np.linspace(X_raw.min() - 0.5, X_raw.max() + 0.5, 100)y_plot = w_ols[0] + w_ols[1] * x_plotplt.scatter(X_raw[:, 0], y, alpha=0.6, label='Data')plt.plot(x_plot, y_plot, 'r-', lw=2, label='OLS fit')plt.title('Linear Regression via Normal Equations', fontsize=12, fontweight='bold')plt.xlabel('Feature'); plt.ylabel('Target')plt.legend(); plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Ridge Regression

---

Add regularization: (XᵀX + λI)w = Xᵀy. Always solvable via Cholesky.


In [ ]:
import numpy as npdef ridge_regression(X, y, lam):    n = X.shape[1]    AtA = X.T @ X + lam * np.eye(n)    Atb = X.T @ y    return np.linalg.solve(AtA, Atb)lam = 1.0w_ridge = ridge_regression(X, y, lam)w_ols = np.linalg.solve(X.T @ X, X.T @ y)print(f"OLS:  {w_ols}")print(f"Ridge (λ={lam}): {w_ridge}")print("Ridge shrinks weights toward zero.")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does partial pivoting do?
> **Explanation:** Swaps rows to use the largest available pivot — prevents division by tiny numbers.

**Q2:** When is Cholesky applicable?
> **Explanation:** Only for symmetric positive definite matrices (covariance matrices, kernel matrices).

**Q3:** What does ridge regression add?
> **Explanation:** λI to XᵀX — improves conditioning and shrinks weights toward zero.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Solve a 3×3 system with Gaussian elimination, LU, and np.linalg.solve; verify agreement.

**Exercise 2:** Create a nearly singular matrix; compute its condition number; show regularization helps.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Gaussian elimination | Row reduce | O(n³) to triangular form, then back substitution |
| Partial pivoting | Swap rows for stability | Use largest available pivot to avoid tiny divisors |
| LU decomposition | A = L·U | Factor once, reuse for multiple right-hand sides |
| Cholesky | A = L·Lᵀ | For symmetric positive definite matrices — half the cost of LU |
| Least squares | Best fit when exact impossible | Minimize ||Ax − b||² — the normal equations AᵀAx = Aᵀb |
| Ridge regression | Regularized least squares | (XᵀX + λI)w = Xᵀy — shrinks weights toward zero |
| Condition number | σ_max / σ_min | Sensitivity to input perturbations — lose log₁₀(κ) digits |


# Further Reading

---

- [Gilbert Strang: Linear Algebra](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/)
- [Trefethen & Bau: Numerical Linear Algebra](https://people.maths.ox.ac.uk/trefethen/text.html)


# Next Steps

---

When you're ready, continue to Lesson 18: Convex Optimization.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
